
# UK Band Cultural Impact Analysis

This notebook explores the cultural influence of famous UK bands by city, using real Spotify streaming data.  
By combining each band's follower count with their home city's population, we can estimate **per-capita cultural productivity**.

We'll use the Spotify API to pull artist data, cache it for future runs, and visualize two key metrics:

- **Average listeners per million residents** by city
- **Top band per city** ranked by per-capita impact



In [ ]:
import os
from dotenv import load_dotenv
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
from datetime import datetime
import re
import json
from pathlib import Path
import requests
import pandas as pd


In [10]:
# empty cached Spotify responses when needed
for cache_file in ["spotify_band_data_cache.json", "spotify_band_id_cache.json"]:
    if os.path.exists(cache_file):
        os.remove(cache_file)


In [3]:
# Authenticate using credentials from .env
load_dotenv(".env")
sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(
    client_id=os.getenv("SPOTIPY_CLIENT_ID"),
    client_secret=os.getenv("SPOTIPY_CLIENT_SECRET")
))
ID_CACHE_FILE = "spotify_band_id_cache.json"
DATA_CACHE_FILE = "spotify_band_data_cache.json"
SPOTSCRAPER_BASE_URL = "https://api.spotscraper.com/v1"


### 🎶 Band Selection
We start from curated lists of bands associated with major UK cities.  
This introduces **selection bias** as I focus on bands that I remember.  

In [4]:
bands_by_city = {
    "London": ["The Rolling Stones", "The Clash", "Blur", "Coldplay", "Florence and the Machine"],
    "Birmingham": ["Black Sabbath", "ELO", "Duran Duran", "UB40", "Editors"],
    "Manchester": ["Oasis", "The Smiths", "New Order", "The Stone Roses", "The 1975"],
    "Glasgow": ["Simple Minds", "Primal Scream", "Belle and Sebastian", "Franz Ferdinand", "CHVRCHES"],
    "Liverpool": ["The Beatles", "Echo & the Bunnymen", "Frankie Goes to Hollywood", "The La's", "The Wombats"],
    "Leeds": ["Kaiser Chiefs", "Soft Cell", "Alt-J", "Sisters of Mercy", "The Wedding Present"],
    "Sheffield": ["Def Leppard", "The Human League", "Pulp", "Arctic Monkeys", "Bring Me The Horizon"],
    "Bradford": ["Smokie", "Tasmin Archer", "Kiki Dee", "Ivyrise", "The Cult"],  # The Cult formed nearby
    "Bristol": ["Massive Attack", "Portishead", "Tricky", "Idles", "Kosheen"],
    "Nottingham": ["Jake Bugg", "Dog Is Dead", "London Grammar", "Ten Years After", "Sleaford Mods"]
}



### City Population Data

To adjust for differences in city size, we use **Built-Up Area (BUA)** population figures from the **Office for National Statistics (ONS) 2021 Census**.  
The BUA measure captures the **continuous urban footprint** of a city, rather than its administrative boundaries.  

This approach is more appropriate for cultural analysis because:
- Many UK city councils cover relatively small areas (e.g. "Manchester" city proper is only ~550k residents, but the BUA is ~2.9M).  
- Music “scenes” typically draw from the wider metro region, not just the city center.  

**Sources:**  
- ONS Census 2021 (England & Wales)  
- National Records of Scotland Census 2021 (for Glasgow)  
- Data compiled via [citypopulation.de](https://www.citypopulation.de/en/uk/cities/) for accessibility.  


In [5]:
city_pop = {
    "London": 9540576,      # Greater London BUA, ONS 2021
    "Birmingham": 2607437,  # West Midlands BUA, ONS 2021
    "Manchester": 2867826,  # Greater Manchester BUA, ONS 2021
    "Glasgow": 1673332,     # Greater Glasgow BUA, Scotland Census 2021
    "Liverpool": 1525690,   # Liverpool BUA, ONS 2021
    "Leeds": 1903228,       # Leeds BUA, ONS 2021
    "Sheffield": 1569000,   # Sheffield BUA, ONS 2021
    "Bradford": 1379872,    # Bradford BUA, ONS 2021
    "Bristol": 712225,      # Bristol BUA, ONS 2021
    "Nottingham": 919484    # Nottingham BUA, ONS 2021
}

### Get Spotify Data

Spotify provides **followers** and **popularity scores**, but not total streams.  
- Followers ≈ long-term audience.  
- Popularity ≈ current listening activity.  
Limitations: Spotify data skews toward younger and digital-native audiences; legacy bands may be underrepresented.  

In [6]:
# Fetch Spotify artist IDs and cache
def _normalize_name(value):
    return re.sub(r'[^a-z0-9]+', '', value.lower()) if value else ''

def _pick_best_match(band, candidates):
    target = _normalize_name(band)
    for artist in candidates:
        if _normalize_name(artist['name']) == target:
            return artist, 'exact'
    for artist in candidates:
        normalized = _normalize_name(artist['name'])
        if normalized in target or target in normalized:
            return artist, 'approximate'
    return (candidates[0], 'fallback') if candidates else (None, None)

def fetch_spotify_ids():
    records = []
    timestamp = datetime.now().strftime('%Y-%m-%d')
    for city, bands in bands_by_city.items():
        for band in bands:
            try:
                result = sp.search(q=f'artist:{band}', type='artist', limit=5)
                items = result.get('artists', {}).get('items', [])
                if not items:
                    print(f'No Spotify match for {band}')
                    continue
                artist, match_quality = _pick_best_match(band, items)
                if artist is None:
                    print(f'Unable to select Spotify artist for {band}')
                    continue
                if match_quality != 'exact':
                    print(f"{band}: using {artist["name"]} ({match_quality})")
                records.append({
                    'band': band,
                    'city': city,
                    'spotify_id': artist['id'],
                    'spotify_name': artist['name'],
                    'match_quality': match_quality,
                    'extracted_at': timestamp
                })
            except Exception as exc:
                print(f'Error fetching Spotify ID for {band}: {exc}')
    with open(ID_CACHE_FILE, 'w') as f:
        json.dump(records, f, indent=2)
    return records

def load_spotify_ids():
    if Path(ID_CACHE_FILE).exists():
        with open(ID_CACHE_FILE) as f:
            return json.load(f)
    return fetch_spotify_ids()

spotify_id_records = load_spotify_ids()
print(f'Loaded {len(spotify_id_records)} artist IDs')
spotify_id_records[:3]


Loaded 50 artist IDs


[{'band': 'The Rolling Stones',
  'city': 'London',
  'spotify_id': '22bE4uQ6baNwSHPVcDxLCe',
  'spotify_name': 'The Rolling Stones',
  'match_quality': 'exact',
  'extracted_at': '2025-09-20'},
 {'band': 'The Clash',
  'city': 'London',
  'spotify_id': '3RGLhK1IP9jnYFH4BRFJBS',
  'spotify_name': 'The Clash',
  'match_quality': 'exact',
  'extracted_at': '2025-09-20'},
 {'band': 'Blur',
  'city': 'London',
  'spotify_id': '7MhMgCo0Bl0Kukl93PZbYS',
  'spotify_name': 'Blur',
  'match_quality': 'exact',
  'extracted_at': '2025-09-20'}]

### Get Monthly streams
I found this API from SpotScraper at https://www.spotscraper.com/ that seems to provide what I want

In [7]:
# Fetch monthly listeners and followers from SpotScraper
def _extract_spotscraper_stats(payload):
    if not isinstance(payload, dict):
        return None, None, None
    data = payload.get('data', payload)
    if not isinstance(data, dict):
        return None, None, None
    stats = data.get('statistics') or data.get('stats') or {}
    if not isinstance(stats, dict):
        stats = {}
    monthly_listeners = (
        stats.get('monthlyListeners')
        or stats.get('monthly_listeners')
    )
    followers = stats.get('followers')
    world_rank = stats.get('worldRank') or stats.get('world_rank')
    if monthly_listeners is None:
        monthly_listeners = data.get('monthlyListeners') or data.get('monthly_listeners')
    if followers is None:
        followers = data.get('followers') or data.get('followers_total') or data.get('followersTotal')
    return monthly_listeners, followers, world_rank

def _coerce_number(value):
    if value is None:
        return None
    if isinstance(value, (int, float)):
        return value
    try:
        return float(value)
    except (TypeError, ValueError):
        return None

def fetch_spotscraper_metrics(records):
    api_key = os.getenv('SPOTSCRAPER_API_KEY')
    if not api_key:
        raise RuntimeError('Missing SPOTSCRAPER_API_KEY environment variable')
    headers = {
        'X-API-Key': api_key,
        'Accept': 'application/json'
    }
    stats_timestamp = datetime.now().strftime('%Y-%m-%d')
    metrics = []
    for record in records:
        artist_id = record['spotify_id']
        try:
            response = requests.get(
                f"{SPOTSCRAPER_BASE_URL}/artists/{artist_id}",
                headers=headers,
                timeout=10
            )
            response.raise_for_status()
            raw_monthly, raw_followers, world_rank = _extract_spotscraper_stats(response.json())
            monthly_listeners = _coerce_number(raw_monthly)
            if monthly_listeners is None:
                print(f"Missing monthly listeners for {record['band']} (raw value: {raw_monthly})")
            if monthly_listeners is None or raw_monthly is None:
                print(f"Missing monthly listeners for {record['band']} (raw value: {raw_monthly})")
            followers = _coerce_number(raw_followers)
            if followers is None:
                print(f"Missing followers for {record['band']} (raw value: {raw_followers})")
            if followers is None or raw_followers is None:
                print(f"Missing followers for {record['band']} (raw value: {raw_followers})")
            if monthly_listeners is None or followers is None:
                raise ValueError(
                    f"SpotScraper response missing metrics for {record['band']} (monthly_listeners={raw_monthly}, followers={raw_followers})"
                )
            metrics.append({
                'band': record['band'],
                'city': record['city'],
                'spotify_id': artist_id,
                'spotify_name': record.get('spotify_name'),
                'match_quality': record.get('match_quality'),
                'monthly_listeners': monthly_listeners,
                'monthly_listeners_m': round(monthly_listeners / 1_000_000, 2),
                'followers': followers,
                'world_rank': world_rank,
                'stats_extracted_at': stats_timestamp
            })
        except requests.HTTPError as exc:
            print(f"HTTP error for {record['band']}: {exc}")
        except ValueError:
            raise
        except Exception as exc:
            print(f"Error fetching SpotScraper metrics for {record['band']}: {exc}")
    return metrics

def load_spotscraper_metrics(records):
    desired_ids = {record['spotify_id'] for record in records if record.get('spotify_id')}
    existing = []
    if Path(DATA_CACHE_FILE).exists():
        with open(DATA_CACHE_FILE) as f:
            try:
                existing = json.load(f)
            except json.JSONDecodeError:
                print('Warning: SpotScraper cache is corrupted. Rebuilding…')
                existing = []
    existing_by_id = {}
    for entry in existing:
        spotify_id = entry.get('spotify_id')
        if spotify_id and spotify_id in desired_ids:
            existing_by_id[spotify_id] = entry
    missing_records = [
        record
        for record in records
        if record['spotify_id'] not in existing_by_id
    ]
    if missing_records:
        print(f"Fetching SpotScraper metrics for {len(missing_records)} missing artists")
        new_entries = fetch_spotscraper_metrics(missing_records)
        for entry in new_entries:
            spotify_id = entry.get('spotify_id')
            if spotify_id and spotify_id in desired_ids:
                existing_by_id[spotify_id] = entry
    combined = [
        existing_by_id[spotify_id]
        for spotify_id in desired_ids
        if spotify_id in existing_by_id
    ]
    combined.sort(key=lambda item: (item.get('city') or '', item.get('band') or ''))
    with open(DATA_CACHE_FILE, 'w') as f:
        json.dump(combined, f, indent=2)
    missing_after_fetch = [
        record
        for record in records
        if record['spotify_id'] not in {entry['spotify_id'] for entry in combined}
    ]
    if missing_after_fetch:
        print(f"Warning: no SpotScraper metrics for {[record['band'] for record in missing_after_fetch]}")
    return combined

spotify_metrics = load_spotscraper_metrics(spotify_id_records)
print(f"Loaded SpotScraper metrics for {len(spotify_metrics)} artists")
spotify_metrics[:3]


Fetching SpotScraper metrics for 1 missing artists
Loaded SpotScraper metrics for 50 artists


[{'band': 'Black Sabbath',
  'city': 'Birmingham',
  'spotify_id': '5M52tdBnJaKSvOpJGz8mfZ',
  'spotify_name': 'Black Sabbath',
  'match_quality': 'exact',
  'monthly_listeners': 19891349,
  'monthly_listeners_m': 19.89,
  'followers': 9635964,
  'world_rank': 389,
  'stats_extracted_at': '2025-09-20'},
 {'band': 'Duran Duran',
  'city': 'Birmingham',
  'spotify_id': '0lZoBs4Pzo7R89JM9lxwoT',
  'spotify_name': 'Duran Duran',
  'match_quality': 'exact',
  'monthly_listeners': 14260413,
  'monthly_listeners_m': 14.26,
  'followers': 3525676,
  'world_rank': None,
  'stats_extracted_at': '2025-09-20'},
 {'band': 'ELO',
  'city': 'Birmingham',
  'spotify_id': '7jefIIksOi1EazgRTfW2Pk',
  'spotify_name': 'Electric Light Orchestra',
  'match_quality': 'exact',
  'monthly_listeners': 14861430,
  'monthly_listeners_m': 14.86,
  'followers': 4425636,
  'world_rank': None,
  'stats_extracted_at': '2025-09-20'}]

In [18]:
# Build DataFrame with SpotScraper metrics
df = pd.DataFrame(spotify_metrics)
df['listeners_per_capita'] = df.apply(
    lambda row: round(row['monthly_listeners'] / city_pop[row['city']],2) if row.get('monthly_listeners') is not None else None,
    axis=1
)
print(df.shape)
df.head()


(50, 11)


,band,city,spotify_id,spotify_name,match_quality,monthly_listeners,monthly_listeners_m,followers,world_rank,stats_extracted_at,listeners_per_capita
0,Black Sabbath,Birmingham,5M52tdBnJaKSvOpJGz8mfZ,Black Sabbath,exact,19891349,19.89,9635964,389.0,2025-09-20,7.63
1,Duran Duran,Birmingham,0lZoBs4Pzo7R89JM9lxwoT,Duran Duran,exact,14260413,14.26,3525676,NaN,2025-09-20,5.47
2,ELO,Birmingham,7jefIIksOi1EazgRTfW2Pk,Electric Light Orchestra,exact,14861430,14.86,4425636,NaN,2025-09-20,5.70
3,Editors,Birmingham,6e9wIFWhBPHLE9bXK8gtBI,Editors,exact,808289,0.81,764162,NaN,2025-09-20,0.31
4,UB40,Birmingham,69MEO1AADKg1IZrq2XLzo5,UB40,exact,9078938,9.08,2614020,NaN,2025-09-20,3.48


### Interpreting the Charts

- **Average Spotify Reach per Band (Per Capita)**  
  This chart shows the *typical* per-capita reach of bands from each city.  
  It averages the Spotify listeners (per million residents) across all bands in the dataset for a given city.  
  → Example: *“On average, a band from Liverpool attracts around X Spotify listeners per million Liverpudlians.”*  

- **Top Band per City (Per Capita)**  
  This chart highlights the *single band with the highest per-capita reach* from each city.  
  Naturally, these numbers are higher than the averages, since they represent the standout success story for each location.  
  → Example: *“Liverpool’s leading band (The Beatles) attracts far more per-capita listeners than the city’s average band.”*  

Together, the two views contrast the **typical case vs the exceptional case**, showing both the baseline strength of a city’s music scene and its most successful export.

In [ ]:
import altair as alt
import pandas as pd

# Prepare data for Altair
city_avg = df.groupby("city")["listeners_per_capita"].mean().sort_values(ascending=False).reset_index()
city_avg.columns = ["city", "avg_listeners_per_capita"]

# Altair chart
chart = (
    alt.Chart(city_avg)
    .mark_bar(color="gray")
    .encode(
        y=alt.Y("city:N", sort='-x', title="City"),
        x=alt.X("avg_listeners_per_capita:Q", title="Avg. Spotify Listeners per Resident"),
        tooltip=["city", "avg_listeners_per_capita"]
    )
    .properties(
        width=600,
        height=400,
        title="Average Spotify Listeners per Resident (by City)"
    )
)

# Add text labels on bars
text = (
    alt.Chart(city_avg)
    .mark_text(align="left", baseline="middle", dx=3)
    .encode(
        y=alt.Y("city:N", sort='-x'),
        x="avg_listeners_per_capita:Q",
        text=alt.Text("avg_listeners_per_capita:Q", format=".1f")
    )
)

chart + text

alt.LayerChart(...)

In [ ]:
import altair as alt

# Prepare data
top_band_per_city = (
    df.sort_values("listeners_per_capita", ascending=False)
      .groupby("city").first()
      .reset_index()
      .sort_values("listeners_per_capita", ascending=False)
)

# Create a label column: "Band (Value)"
top_band_per_city["label"] = top_band_per_city.apply(
    lambda row: f"{row['band']} ({row['listeners_per_capita']:.1f})", axis=1
)

# Chart
bars = (
    alt.Chart(top_band_per_city)
    .mark_bar(color="gray")
    .encode(
        y=alt.Y("city:N", sort='-x', title="City"),
        x=alt.X("listeners_per_capita:Q", title="Spotify Listeners per Resident"),
        tooltip=["city", "band", "listeners_per_capita"]
    )
    .properties(
        width=600,
        height=400,
        title={
            "text": ["Top Band per City by Normalized Spotify Reach"],
            "subtitle": ["Which band from each city attracts the most Spotify listeners per resident?"],
            "anchor": "start"
        }
    )
)

# Labels with band name + value
text = (
    alt.Chart(top_band_per_city)
    .mark_text(align="left", baseline="middle", dx=3)
    .encode(
        y=alt.Y("city:N", sort='-x'),
        x="listeners_per_capita:Q",
        text="label:N"
    )
)

bars + text

alt.LayerChart(...)

### City Population vs. Average Spotify Reach

This scatterplot compares **city population size** with the **average per-band Spotify reach (per million residents)**.  

- The horizontal axis shows the population of each city (in millions).  
- The vertical axis shows the average per-capita listeners for bands from that city.  
- Each point represents a city, labeled by name.  

**How to read it:**  
- Cities further right are larger (London, Manchester).  
- Cities higher up have bands that, on average, attract more per-capita listeners.  
- Outliers reveal places that *punch above their weight* musically — smaller cities with bands that have unusually strong per-capita visibility.  

In [ ]:
import altair as alt
import math

# Prepare data
city_avg = df.groupby("city")["listeners_per_capita"].mean()
pop_series = pd.Series(city_pop)

scatter_df = pd.DataFrame({
    "City": city_avg.index,
    "PopulationMillions": pop_series / 1_000_000,  # convert to millions
    "Impact": city_avg
}).dropna().reset_index(drop=True)

population_tick_max = max(1, math.ceil(scatter_df["PopulationMillions"].max()))
population_ticks = list(range(0, population_tick_max + 1))

# Scatter plot
points = (
    alt.Chart(scatter_df)
    .mark_circle(size=120, color="gray")
    .encode(
        x=alt.X(
            "PopulationMillions:Q",
            title="City Population (millions)",
            axis=alt.Axis(values=population_ticks, tickMinStep=1, format=".0f")
        ),
        y=alt.Y("Impact:Q", title="Avg. Spotify Listeners per Resident", axis=alt.Axis(format=".0f")),
        tooltip=["City", "PopulationMillions", "Impact"]
    )
    .properties(
        width=600,
        height=400,
        title={
            "text": ["City Population vs. Average Per-Band Spotify Reach"],
            "subtitle": ["Comparing city size (in millions) with per-capita visibility of its bands"],
            "anchor": "start"
        }
    )
)

# Labels
labels = (
    alt.Chart(scatter_df)
    .mark_text(align="left", baseline="middle", dx=7, dy=-7, fontSize=11)
    .encode(
        x="PopulationMillions:Q",
        y="Impact:Q",
        text="City:N"
    )
)

points + labels



alt.LayerChart(...)


## Objective Top UK Acts (MusicBrainz + Spotify)

To balance the subjective city lists above, the next cells pull a broader catalogue of UK artists directly from MusicBrainz, map them to Spotify, and surface the top acts by follower count.

**Workflow**
- Query MusicBrainz for UK-based artists (groups and solo acts) with their recorded home area.
- Reuse the existing Spotify search helpers to align each act with a Spotify artist profile.
- Rank the results by Spotify follower counts and inspect the top 100.

> MusicBrainz requires a descriptive User-Agent string; update the constant below with your contact details before heavy use.


In [144]:
import functools
import hashlib
import json
import time
from dataclasses import dataclass
from typing import Iterable, Optional
from pathlib import Path

import pandas as pd
import pycountry

MUSICBRAINZ_ARTIST_ENDPOINT = "https://musicbrainz.org/ws/2/artist"
MUSICBRAINZ_AREA_ENDPOINT = "https://musicbrainz.org/ws/2/area"
MUSICBRAINZ_USER_AGENT = os.getenv("MUSICBRAINZ_USER_AGENT", "python-uk-bands/1.0 (contact: info@danielpradilla.info)")
MUSICBRAINZ_HEADERS = {"User-Agent": MUSICBRAINZ_USER_AGENT}

DEFAULT_TYPES = ("Group", "Person")
MUSICBRAINZ_BATCH_LIMIT = 100

ALLOWED_CITY_AREA_TYPES = {
    "city",
    "town",
    "village",
    "municipality",
    "borough",
    "metropolitan borough",
    "district borough",
    "london borough",
    "metropolitan area",
    "urban district",
    "civil parish",
    "locality",
    "region/city",
    "suburb",
    "city district",
    "commune",
}

DISALLOWED_AREA_TYPES = {
    "country",
    "state",
    "region",
    "district",
    "county",
    "province",
    "subdivision",
    "unitary authority",
}


@dataclass
class MBFetchConfig:
    query: str = "country:GB"
    include_types: Iterable[str] = DEFAULT_TYPES
    min_relevance: Optional[int] = 65
    batch_size: int = 100
    max_artists: int = 2000
    throttle_seconds: float = 0.5
    request_attempts: int = 3
    max_offset_pages: Optional[int] = 5
    sleep_seconds: float = 1.0

    def query_string(self) -> str:
        type_clause = " OR \n".join(f"type:{t}" for t in self.include_types) if self.include_types else ""
        return f"{self.query} AND ({type_clause})" if type_clause else self.query


@dataclass
class SpotifyFetchConfig:
    cache_slug: str
    search_limit: int = 5
    sleep_seconds: float = 0.1


# ---------------------------------------------------------------------------
# Cache helpers
# ---------------------------------------------------------------------------

_CACHE_DIR = Path('.')


def _cache_slug(parts: Iterable[str]) -> str:
    key = '||'.join(parts)
    return hashlib.sha1(key.encode('utf-8')).hexdigest()[:10]


def _musicbrainz_cache_path(config: MBFetchConfig) -> Path:
    slug = _cache_slug([
        config.query_string(),
        ','.join(config.include_types),
        str(config.min_relevance),
        str(config.max_artists),
        str(config.batch_size),
    ])
    return _CACHE_DIR / f"musicbrainz_artists_{slug}.json"


def _spotify_cache_path(spotify_config: SpotifyFetchConfig) -> Path:
    return _CACHE_DIR / f"spotify_artists_{spotify_config.cache_slug}.json"


# ---------------------------------------------------------------------------
# Config utilities
# ---------------------------------------------------------------------------


def _ensure_config(config: Optional[MBFetchConfig], fetch_kwargs: dict) -> MBFetchConfig:
    if config is not None and fetch_kwargs:
        raise ValueError("Provide either an MBFetchConfig or keyword arguments, not both.")
    if config is None:
        config = MBFetchConfig(**fetch_kwargs)
    # normalise include_types to tuple for consistent hashing
    if not isinstance(config.include_types, tuple):
        config.include_types = tuple(config.include_types)
    return config


# ---------------------------------------------------------------------------
# MusicBrainz request helpers
# ---------------------------------------------------------------------------


def _musicbrainz_request(params: dict, *, throttle: float, attempts: int, offset: int, collected: int) -> dict:
    for attempt in range(attempts):
        if throttle and (offset > 0 or collected > 0 or attempt > 0):
            time.sleep(throttle * (attempt + 1))
        try:
            response = requests.get(
                MUSICBRAINZ_ARTIST_ENDPOINT,
                params=params,
                headers=MUSICBRAINZ_HEADERS,
                timeout=15,
            )
            response.raise_for_status()
            return response.json()
        except requests.HTTPError as exc:
            status = exc.response.status_code if exc.response else None
            if status and status >= 500 and attempt < attempts - 1:
                time.sleep(max(throttle, 0.5) * (attempt + 1))
                continue
            raise
    raise RuntimeError(f"MusicBrainz request failed after {attempts} attempts (offset={offset})")


def _resolve_country_name(artist: dict) -> Optional[str]:
    area = artist.get("area") or {}
    area_type = (area.get("type") or "").lower()
    area_name = (area.get("name") or "").strip()
    if area_name and area_type == "country":
        return area_name
    country_code = (artist.get("country") or "").strip()
    if country_code:
        try:
            return pycountry.countries.lookup(country_code).name
        except LookupError:
            pass
    return area_name or None


@functools.lru_cache(maxsize=2048)
def _fetch_area_details(area_id: str, attempts: int = 3, backoff_base: float = 1.0) -> dict:
    for attempt in range(attempts):
        try:
            params = {"fmt": "json", "inc": "area-rels"}
            response = requests.get(
                MUSICBRAINZ_AREA_ENDPOINT + f"/{area_id}",
                params=params,
                headers=MUSICBRAINZ_HEADERS,
                timeout=15,
            )
            response.raise_for_status()
            return response.json()
        except requests.HTTPError as exc:
            status = exc.response.status_code if exc.response else None
            if status and status >= 500 and attempt < attempts - 1:
                time.sleep(backoff_base * (attempt + 1))
                continue
            raise
    raise RuntimeError(f"Unable to fetch area details for {area_id} after {attempts} attempts")


def _resolve_city_from_area(area_dict: dict, country_name: Optional[str], depth: int = 0) -> Optional[str]:
    if not isinstance(area_dict, dict):
        return None
    area_id = area_dict.get("id")
    area_name = (area_dict.get("name") or "").strip()
    area_type = (area_dict.get("type") or "").lower()

    def _is_country_match(name: str) -> bool:
        return bool(name) and country_name and name.lower() == country_name.strip().lower()

    if area_name and not _is_country_match(area_name):
        if area_type in ALLOWED_CITY_AREA_TYPES or (not area_type and country_name):
            return area_name
        if not area_type and not country_name:
            return area_name
    if area_type and area_type in DISALLOWED_AREA_TYPES:
        pass
    elif area_name and not area_type and not _is_country_match(area_name):
        return area_name

    if not area_id or depth > 6:
        return None
    try:
        details = _fetch_area_details(area_id)
    except requests.HTTPError as exc:
        print(f"Warning: unable to fetch area details for {area_name or area_id}: {exc}")
        return None

    candidate_name = (details.get("name") or area_name).strip()
    candidate_type = (details.get("type") or area_type).lower()
    if candidate_name and not _is_country_match(candidate_name):
        if candidate_type in ALLOWED_CITY_AREA_TYPES or (not candidate_type and country_name):
            return candidate_name
        if not candidate_type and not country_name:
            return candidate_name
    if candidate_type and candidate_type in DISALLOWED_AREA_TYPES:
        pass

    for relation in details.get("relations", []):
        if relation.get("type") != "part of":
            continue
        direction = relation.get("direction")
        if direction and direction.lower() == "forward":
            continue
        resolved = _resolve_city_from_area(relation.get("area"), country_name, depth + 1)
        if resolved:
            return resolved
    return None


def _extract_musicbrainz_city(artist: dict, country_name: Optional[str]) -> tuple[Optional[str], Optional[dict]]:
    city_from_area = _resolve_city_from_area(artist.get("area"), country_name)
    if city_from_area:
        return city_from_area, None
    city_from_begin = _resolve_city_from_area(artist.get("begin-area"), country_name)
    if city_from_begin:
        return city_from_begin, None
    begin_area_name = ((artist.get("begin-area") or {}).get("name") or "").strip()
    if begin_area_name:
        if country_name and begin_area_name.lower() == country_name.strip().lower():
            return None, {"reason": "begin_area_matches_country", "label": begin_area_name or country_name}
        return begin_area_name, None
    return None, None


def _filter_by_relevance(artists: list[dict], min_relevance: Optional[int]) -> list[dict]:
    if min_relevance is None:
        return artists
    return [artist for artist in artists if (artist.get("score") or 0) >= min_relevance]


# ---------------------------------------------------------------------------
# Public helpers
# ---------------------------------------------------------------------------


def estimate_musicbrainz_count(config: Optional[MBFetchConfig] = None, **fetch_kwargs) -> Optional[int]:
    config = _ensure_config(config, fetch_kwargs)
    params = {
        "query": config.query_string(),
        "fmt": "json",
        "limit": MUSICBRAINZ_BATCH_LIMIT,
        "offset": 0,
    }
    payload = _musicbrainz_request(
        params,
        throttle=config.throttle_seconds,
        attempts=config.request_attempts,
        offset=0,
        collected=0,
    )
    total_count = payload.get("count")
    if total_count is None or config.min_relevance is None:
        return total_count
    artists = payload.get("artists", [])
    if not artists:
        return total_count
    ratio = len(_filter_by_relevance(artists, config.min_relevance)) / len(artists)
    return int(total_count * ratio)


def preview_musicbrainz_results(
    config: Optional[MBFetchConfig] = None,
    *,
    limit: int = 10,
    offset: int = 0,
    **fetch_kwargs,
) -> dict:
    config = _ensure_config(config, fetch_kwargs)
    if limit > MUSICBRAINZ_BATCH_LIMIT:
        raise ValueError(f"MusicBrainz API limit is {MUSICBRAINZ_BATCH_LIMIT} per request")
    params = {
        "query": config.query_string(),
        "fmt": "json",
        "limit": limit,
        "offset": offset,
    }
    payload = _musicbrainz_request(
        params,
        throttle=config.throttle_seconds,
        attempts=config.request_attempts,
        offset=offset,
        collected=0,
    )
    artists = payload.get("artists", [])
    filtered = _filter_by_relevance(artists, config.min_relevance)
    return {"raw": payload, "filtered": pd.DataFrame(filtered)}


def fetch_musicbrainz_artists(config: Optional[MBFetchConfig] = None, **fetch_kwargs) -> pd.DataFrame:
    config = _ensure_config(config, fetch_kwargs)
    batch_size = min(config.batch_size, MUSICBRAINZ_BATCH_LIMIT)
    collected: list[dict] = []
    offset = 0
    total_available = None

    while len(collected) < config.max_artists:
        remaining = config.max_artists - len(collected)
        params = {
            "query": config.query_string(),
            "fmt": "json",
            "limit": min(batch_size, remaining),
            "offset": offset,
        }
        payload = _musicbrainz_request(
            params,
            throttle=config.throttle_seconds,
            attempts=config.request_attempts,
            offset=offset,
            collected=len(collected),
        )

        if total_available is None:
            total_available = payload.get("count")

        artists = _filter_by_relevance(payload.get("artists", []), config.min_relevance)
        if not artists:
            break

        for artist in artists:
            country_name = _resolve_country_name(artist)
            city_name, skip_info = _extract_musicbrainz_city(artist, country_name)
            if skip_info:
                print(f"Skipping {artist.get('name')} – begin-area equals country ({skip_info.get('label')})")
                continue
            collected.append({
                "musicbrainz_id": artist.get("id"),
                "name": artist.get("name"),
                "type": artist.get("type"),
                "disambiguation": artist.get("disambiguation"),
                "city": city_name,
                "country": country_name,
                "life_span_begin": (artist.get("life-span") or {}).get("begin"),
                "life_span_end": (artist.get("life-span") or {}).get("ended"),
                "score": artist.get("score"),
            })
            if len(collected) >= config.max_artists:
                break

        if len(artists) < params["limit"]:
            break

        offset += batch_size
        if config.max_offset_pages is not None and (offset // batch_size) >= config.max_offset_pages:
            break

        if config.sleep_seconds:
            time.sleep(config.sleep_seconds)

    df = pd.DataFrame(collected).drop_duplicates(subset=["musicbrainz_id"]).reset_index(drop=True)
    if total_available is not None:
        df.attrs["musicbrainz_total_count"] = total_available
    df["seq"] = range(1, len(df) + 1)
    return df


def load_musicbrainz_artists(config: Optional[MBFetchConfig] = None, **fetch_kwargs) -> pd.DataFrame:
    config = _ensure_config(config, fetch_kwargs)
    cache_path = _musicbrainz_cache_path(config)
    if cache_path.exists():
        try:
            records = json.loads(cache_path.read_text())
            return pd.DataFrame(records)
        except json.JSONDecodeError:
            print("Warning: MusicBrainz cache is corrupted. Re-fetching…")
    acts = fetch_musicbrainz_artists(config)
    cache_path.write_text(json.dumps(acts.to_dict(orient="records"), indent=2))
    return acts


def load_spotify_artists(
    artists_df: pd.DataFrame,
    config: Optional[MBFetchConfig] = None,
    *,
    search_limit: int = 5,
    sleep_seconds: float = 0.1,
) -> pd.DataFrame:
    if config is None:
        config = MBFetchConfig()
    cache_slug = _cache_slug([
        config.query_string(),
        ','.join(config.include_types),
        str(config.min_relevance),
        str(search_limit),
    ])
    cache_path = _spotify_cache_path(SpotifyFetchConfig(cache_slug, search_limit, sleep_seconds))
    if cache_path.exists():
        try:
            cached = pd.read_json(cache_path)
            cached = cached[cached["musicbrainz_id"].isin(artists_df["musicbrainz_id"])]
            if not cached.empty:
                return cached
        except ValueError:
            print("Warning: Spotify cache is corrupted. Rebuilding…")

    records = []
    seen_spotify_ids = set()
    for row in artists_df.itertuples():
        query = f"artist:{row.name}"
        try:
            result = sp.search(q=query, type="artist", limit=search_limit)
        except spotipy.SpotifyException as exc:
            print(f"Spotify search failed for {row.name}: {exc}")
            continue
        items = result.get('artists', {}).get('items', [])
        if not items:
            continue
        artist, match_quality = _pick_best_match(row.name, items)
        if artist is None:
            continue
        spotify_id = artist.get('id')
        followers = (artist.get('followers') or {}).get('total')
        if spotify_id in seen_spotify_ids or followers is None:
            continue
        seen_spotify_ids.add(spotify_id)
        records.append({
            "musicbrainz_id": row.musicbrainz_id,
            "name": row.name,
            "city": getattr(row, 'city', None),
            "spotify_id": spotify_id,
            "spotify_name": artist.get('name'),
            "spotify_followers": followers,
            "spotify_popularity": artist.get('popularity'),
            "match_quality": match_quality,
        })
        if sleep_seconds:
            time.sleep(sleep_seconds)

    enriched = pd.DataFrame(records)
    cache_path.write_text(json.dumps(enriched.to_dict(orient="records"), indent=2))
    return enriched



In [ ]:
myconfig = MBFetchConfig(
    query="country:GB",           # any Lucene query, e.g. 'area:"New York"'
    include_types=("Group", "Person"),
    min_relevance=55,
    batch_size=100,
    max_artists=2000,
    throttle_seconds=0.5,
    request_attempts=5,
    max_offset_pages=5,
    sleep_seconds=1.0,
)
preview_musicbrainz_results(myconfig)['raw']

{'created': '2025-09-21T21:31:24.002Z',
 'count': 79759,
 'offset': 0,
 'artists': [{'id': 'b10bbbfc-cf9e-42e0-be17-e2c3e1d2600d',
   'type': 'Group',
   'type-id': 'e431f5f6-b5d2-343d-8b36-72607fffb74b',
   'score': 100,
   'name': 'The Beatles',
   'sort-name': 'Beatles, The',
   'country': 'GB',
   'area': {'id': '8a754a16-0027-3a29-b6d7-2b40ea0481ed',
    'type': 'Country',
    'type-id': '06dd0ae4-8c74-30bb-b43d-95dcedf961de',
    'name': 'United Kingdom',
    'sort-name': 'United Kingdom',
    'life-span': {'ended': None}},
   'begin-area': {'id': 'c249c30e-88ab-4b2f-a745-96a25bd7afee',
    'type': 'City',
    'type-id': '6fd8f29a-3d0a-32fc-980d-ea697b69da78',
    'name': 'Liverpool',
    'sort-name': 'Liverpool',
    'life-span': {'ended': None}},
   'disambiguation': 'UK rock band, “The Fab Four”',
   'isnis': ['0000000121707484'],
   'life-span': {'begin': '1960', 'end': '1970-04-10', 'ended': True},
   'aliases': [{'sort-name': '披头士乐队',
     'type-id': '894afba6-2816-3c24-807

In [113]:
def preview_musicbrainz_city(artist_name, limit=5):
    """Quickly inspect MusicBrainz matches and resolved cities for a single artist.

    Parameters
    ----------
    artist_name : str
        Search string passed to MusicBrainz.
    limit : int
        Maximum number of matches to return.
    """
    params = {
        "query": f"artist:{artist_name}",
        "fmt": "json",
        "limit": limit,
    }
    response = requests.get(
        MUSICBRAINZ_ARTIST_ENDPOINT, params=params, headers=MUSICBRAINZ_HEADERS, timeout=15
    )
    response.raise_for_status()
    artists = response.json().get("artists", [])
    preview_rows = []
    for artist in artists:
        country_name = _resolve_country_name(artist)
        city_name, skip_info = _extract_musicbrainz_city(artist, country_name)
        preview_rows.append({
            "musicbrainz_id": artist.get("id"),
            "name": artist.get("name"),
            "type": artist.get("type"),
            "score": artist.get("score"),
            "country": country_name,
            "resolved_city": city_name,
            "skip_reason": skip_info.get("reason") if skip_info else None,
        })
    return pd.DataFrame(preview_rows)
preview_musicbrainz_city('Joy Division')

,musicbrainz_id,name,type,score,country,resolved_city,skip_reason
0,9a58fda3-f4ed-4080-a3a5-f457aac9fcdd,Joy Division,Group,100,United Kingdom,Salford,None
1,a7c3f0e1-027a-4f1f-800a-c115422f96f3,Pansy Division,Group,75,United States,San Francisco,None
2,eec4a9e9-c213-4811-abe2-1dc4ec16f3eb,Division,Group,75,"Washington, D.C.",None,begin_area_matches_country
3,8eef9799-cd23-4928-8d76-24f3cd7631ba,Division,Person,74,Austria,St. Pölten,None
4,2be61e7e-cae2-4ab5-abd0-641c8b559320,Division,Group,72,Slovenia,None,None


In [134]:
myconfig = MBFetchConfig(
    query="country:GB",
    include_types=("Group", "Person"),
    min_relevance=65,
    batch_size=100,
    max_artists=2000,
    throttle_seconds=0.5,
    request_attempts=3,
    max_offset_pages=50,
    sleep_seconds=1.0,
)

estimated_total = estimate_musicbrainz_count(config=myconfig)
if estimated_total is not None:
    print(f"MusicBrainz estimates {estimated_total} total matching artists")



MusicBrainz estimates 79760 total matching artists


In [135]:
# Fetch a catalogue of artists from MusicBrainz
musicbrainz_artists = load_musicbrainz_artists(config=myconfig)
print(f"Fetched {len(musicbrainz_artists)} artists from MusicBrainz")
total_available = musicbrainz_artists.attrs.get("musicbrainz_total_count")
if total_available is not None:
    print(f"MusicBrainz reports {total_available} total matching artists")
print("City counts:")
print(musicbrainz_artists["city"].value_counts().head(10))
print("Score distribution (top 5):")
print(musicbrainz_artists["score"].value_counts().sort_index(ascending=False).head())
print(f"Fetched {len(musicbrainz_artists)} unique MusicBrainz artists")




Skipping Above & Beyond – begin-area equals country (United Kingdom)
Skipping Monty Python – begin-area equals country (United Kingdom)
Skipping Bucks Fizz – begin-area equals country (United Kingdom)
Skipping The D’Oyly Carte Opera Company – begin-area equals country (United Kingdom)
Skipping I’m Sorry I Haven’t a Clue “Team” – begin-area equals country (United Kingdom)
Skipping Andrew Liles – begin-area equals country (United Kingdom)
Skipping Il Divo – begin-area equals country (United Kingdom)
Skipping African Head Charge – begin-area equals country (United Kingdom)
Skipping Dan Starkey – begin-area equals country (United Kingdom)
Skipping Fischer‐Z – begin-area equals country (United Kingdom)
Skipping Sore Throat – begin-area equals country (United Kingdom)
Fetched 1144 artists from MusicBrainz
MusicBrainz reports 79760 total matching artists
City counts:
city
London        347
Liverpool      31
Manchester     28
Glasgow        26
Birmingham     24
Sheffield      18
Leeds         

In [141]:
musicbrainz_artists[musicbrainz_artists['city']=='Sheffield']


,musicbrainz_id,name,type,disambiguation,city,country,life_span_begin,life_span_end,score,seq
54,7249b899-8db8-43e7-9e6e-22f1e736024e,Def Leppard,Group,None,Sheffield,United Kingdom,1977,None,81,55
82,32f2d6bd-c22b-42cf-a7bc-0c4b48cd2bcb,Joe Cocker,Person,None,Sheffield,United Kingdom,1944-05-20,True,79,83
165,7adaabfb-acfb-47bc-8c7c-59471c2f0db8,The Human League,Group,None,Sheffield,United Kingdom,1977,None,75,166
206,ada7a83c-e3e1-40f1-93f9-3e73dbc9298a,Arctic Monkeys,Group,None,Sheffield,United Kingdom,2002,None,74,207
245,228af6b5-f691-4ed2-a63b-10cc6fcc1144,Thompson Twins,Group,None,Sheffield,United Kingdom,1977,True,73,246
265,87199477-b0df-4ead-84ee-9b54b4abfc3d,ABC,Group,English 80s pop group,Sheffield,United Kingdom,1980,None,73,266
287,2f94016a-3880-4d8c-9af9-0e197ee77189,Moloko,Group,English‐Irish electronic music duo,Sheffield,United Kingdom,1993,True,73,288
299,0796f847-09ca-4526-b17e-44390dd536ba,Heaven 17,Group,"Sheffield, UK ""new romantic"" band",Sheffield,United Kingdom,1980,None,72,300
344,93aaa233-6b5f-41bf-bfe8-70cf1552c218,Babybird,Group,None,Sheffield,United Kingdom,1995,None,72,345
347,439a29ad-f4ef-4857-a7cd-3d4a551056c8,Cabaret Voltaire,Group,None,Sheffield,United Kingdom,1973,True,72,348


In [ ]:
# Map artists to Spotify and rank by follower count
spotify_artists = load_spotify_artists(musicbrainz_artists, config=myconfig, search_limit=5, sleep_seconds=0.2)
print(f"Matched {len(spotify_artists)} artists onto Spotify")




Matched 1120 artists onto Spotify


,musicbrainz_id,name,city,spotify_id,spotify_name,spotify_followers,spotify_popularity,match_quality
0,b8a7c51f-362c-4dcb-a259-bc6e0095f0a6,Ed Sheeran,Halifax,6eUKZXaKkcviH0Ku9w2n3V,Ed Sheeran,122145603,93,exact
1,a7409219-a681-4072-adb2-5285106ce6f2,The The,London,1Xyo4u8uXC1ZmMpatF05PJ,The Weeknd,111165726,97,fallback
2,cc197bad-dc9c-440d-a5b5-d52ba2e14234,Coldplay,London,4gzpq5DPGxSnKTe4SA8HAU,Coldplay,60229558,92,exact
3,0383dadf-2a4e-4d10-a46a-e9e041da8eb3,Queen,London,1dfeR4HaWDbWqFHLkxsg1d,Queen,54868726,86,exact
4,6f1a58bf-9b1b-49cf-a44a-6cefad7ae04f,Dua Lipa,London,6M2wZ9GZgrQXHCFfjv46we,Dua Lipa,46658739,89,exact
5,ada7a83c-e3e1-40f1-93f9-3e73dbc9298a,Arctic Monkeys,Sheffield,7Ln80lUS6He07XvHI8qqHH,Arctic Monkeys,31654001,88,exact
6,b10bbbfc-cf9e-42e0-be17-e2c3e1d2600d,The Beatles,Liverpool,3WrFJ7ztbogyGnTHbHJFl2,The Beatles,30531810,86,exact
7,678d88b2-87b0-403b-b63d-5da7465aecc3,Led Zeppelin,London,36QJpDe2go2KgaRleHCDTp,Led Zeppelin,15861834,80,exact
8,b071f9fa-14b0-4217-8e97-eb41da73f598,The Rolling Stones,London,22bE4uQ6baNwSHPVcDxLCe,The Rolling Stones,15603964,81,exact
9,5421d203-e6c8-4ecd-9997-a46cc8742e86,Man,Merthyr Tydfil,0tmwSHipWxN12fsoLcFU3B,Manuel Turizo,15539510,85,approximate


In [155]:
top_spotify_artists_by_followers = (
    spotify_artists[spotify_artists["match_quality"] == "exact"]
    .dropna(subset=['spotify_followers'])
    .sort_values('spotify_followers', ascending=False)
    .head(200)
    .reset_index(drop=True)
)
top_spotify_artists_by_followers

,musicbrainz_id,name,city,spotify_id,spotify_name,spotify_followers,spotify_popularity,match_quality
0,b8a7c51f-362c-4dcb-a259-bc6e0095f0a6,Ed Sheeran,Halifax,6eUKZXaKkcviH0Ku9w2n3V,Ed Sheeran,122145603,93,exact
1,cc197bad-dc9c-440d-a5b5-d52ba2e14234,Coldplay,London,4gzpq5DPGxSnKTe4SA8HAU,Coldplay,60229558,92,exact
2,0383dadf-2a4e-4d10-a46a-e9e041da8eb3,Queen,London,1dfeR4HaWDbWqFHLkxsg1d,Queen,54868726,86,exact
3,6f1a58bf-9b1b-49cf-a44a-6cefad7ae04f,Dua Lipa,London,6M2wZ9GZgrQXHCFfjv46we,Dua Lipa,46658739,89,exact
4,ada7a83c-e3e1-40f1-93f9-3e73dbc9298a,Arctic Monkeys,Sheffield,7Ln80lUS6He07XvHI8qqHH,Arctic Monkeys,31654001,88,exact
...,...,...,...,...,...,...,...,...
195,04b61dcd-28c9-4a3a-89af-4c290535845b,The Saturdays,London,15qI5w4XJFLRMwOp2VrlD5,The Saturdays,735264,51,exact
196,35723b60-732e-4bd8-957f-320b416e7b7f,Groove Armada,London,67tgMwUfnmqzYsNAtnP6YJ,Groove Armada,734090,60,exact
197,23d9d74d-c95e-46a6-be26-a6075c49747a,Faithless,London,5T4UKHhr4HGIC0VzdZQtAE,Faithless,732478,63,exact
198,dfa715ac-b536-44df-af43-570d3ea3edec,Sugababes,London,7rZNSLWMjTbwdLNskFbzFf,Sugababes,731236,62,exact


In [156]:
city_distribution = (
    top_spotify_artists_by_followers["city"]
    .fillna("Unknown")
    .value_counts()
    .rename_axis("City")
    .reset_index(name="Artists")
)
city_distribution

,City,Artists
0,London,71
1,Manchester,9
2,Birmingham,9
3,Liverpool,8
4,Sheffield,6
...,...,...
80,Haywards Heath,1
81,Leicester,1
82,Cwmaman,1
83,Aberdeen,1


In [ ]:
# Fetch SpotScraper metrics for the Spotify-matched artists
data_cache_slug = _cache_slug([
    myconfig.query_string(),
    ','.join(myconfig.include_types),
    str(myconfig.min_relevance),
    'spotscraper'
])
DATA_CACHE_FILE = f"spotify_spotscraper_{data_cache_slug}.json"

spotscraper_records = []
for row in spotify_artists.itertuples():
    if not row.spotify_id:
        continue
    spotscraper_records.append({
        'band': row.name,
        'city': getattr(row, 'city', None),
        'spotify_id': row.spotify_id,
        'spotify_name': row.spotify_name,
        'match_quality': row.match_quality,
    })

spotscraper_metrics = load_spotscraper_metrics(spotscraper_records)
spotscraper_df = pd.DataFrame(spotscraper_metrics)




Fetching SpotScraper metrics for 1120 missing artists
HTTP error for The Beatles: 503 Server Error: Service Unavailable for url: https://api.spotscraper.com/v1/artists/3WrFJ7ztbogyGnTHbHJFl2
HTTP error for The Rolling Stones: 503 Server Error: Service Unavailable for url: https://api.spotscraper.com/v1/artists/22bE4uQ6baNwSHPVcDxLCe


KeyboardInterrupt: 

In [ ]:
# Top 200 by monthly listeners
spotscraper_top200 = (
    spotscraper_df
    .dropna(subset=['monthly_listeners'])
    .sort_values('monthly_listeners', ascending=False)
    .head(200)
    .reset_index(drop=True)
)

# City distribution
spotscraper_city_dist = (
    spotscraper_top200['city']
    .fillna('Unknown')
    .value_counts()
    .rename_axis('City')
    .reset_index(name='Artists')
)

spotscraper_top200.head(20)


In [ ]:
spotscraper_city_dist
